In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA = Path.cwd().parent / 'data' / 'interim'
FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

features    = pd.read_csv(DATA / 'features.csv')
new_clients = pd.read_csv(DATA / 'new_clients.csv')
new_clients['registration_date'] = pd.to_datetime(new_clients['registration_date'], utc=True)

print(f'features: {features.shape}  churn: {features["churned"].mean()*100:.1f}%')


In [ ]:
# Temporal train/val/test split to prevent look-ahead bias
features = features.merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
features['registration_date'] = pd.to_datetime(features['registration_date'], utc=True)

train_mask = features['registration_date'] < pd.Timestamp('2031-06-01', tz='UTC')
val_mask   = (features['registration_date'] >= pd.Timestamp('2031-06-01', tz='UTC')) & \
             (features['registration_date'] <  pd.Timestamp('2031-08-01', tz='UTC'))
test_mask  = features['registration_date'] >= pd.Timestamp('2031-08-01', tz='UTC')

print(f'Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}')
for name, mask in [('Train',train_mask),('Val',val_mask),('Test',test_mask)]:
    print(f'  Churn {name}: {features.loc[mask,"churned"].mean()*100:.1f}%')


In [ ]:
# Encode categoricals and build feature matrices
DROP_COLS = ['client_id','churned','registration_date']
CAT_COLS  = ['first_location','first_category']

df_train = features[train_mask].copy()
df_val   = features[val_mask].copy()
df_test  = features[test_mask].copy()

le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(df_train[col])
    df_train[col] = le.transform(df_train[col])
    df_val[col]   = df_val[col].map(dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    df_test[col]  = df_test[col].map(dict(zip(le.classes_, le.transform(le.classes_)))).fillna(-1).astype(int)
    le_dict[col]  = le

X_train = df_train.drop(columns=DROP_COLS)
y_train = df_train['churned']
X_val   = df_val.drop(columns=DROP_COLS)
y_val   = df_val['churned']
X_test  = df_test.drop(columns=DROP_COLS)
y_test  = df_test['churned']

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
print(f'Features ({len(X_train.columns)}): {X_train.columns.tolist()}')


In [ ]:
# Logistic Regression (baseline)
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
lr_val_proba  = lr.predict_proba(X_val)[:,1]
lr_test_proba = lr.predict_proba(X_test)[:,1]
lr_val_auc  = roc_auc_score(y_val, lr_val_proba)
lr_test_auc = roc_auc_score(y_test, lr_test_proba)
print(f'LR  val={lr_val_auc:.4f}  test={lr_test_auc:.4f}')


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_val_proba  = rf.predict_proba(X_val)[:,1]
rf_test_proba = rf.predict_proba(X_test)[:,1]
rf_val_auc  = roc_auc_score(y_val, rf_val_proba)
rf_test_auc = roc_auc_score(y_test, rf_test_proba)
print(f'RF  val={rf_val_auc:.4f}  test={rf_test_auc:.4f}')


In [ ]:
# XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='auc', early_stopping_rounds=50, verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_val_proba  = xgb_model.predict_proba(X_val)[:,1]
xgb_test_proba = xgb_model.predict_proba(X_test)[:,1]
xgb_val_auc  = roc_auc_score(y_val, xgb_val_proba)
xgb_test_auc = roc_auc_score(y_test, xgb_test_proba)
print(f'XGB val={xgb_val_auc:.4f}  test={xgb_test_auc:.4f}  best_iter={xgb_model.best_iteration}')


In [ ]:
# Model comparison table
print(f"{'Model':<25} {'Val AUC':>9} {'Test AUC':>9}")
print('-' * 45)
for name, v, t in [('Logistic Regression', lr_val_auc, lr_test_auc),
                    ('Random Forest',       rf_val_auc, rf_test_auc),
                    ('XGBoost',             xgb_val_auc, xgb_test_auc)]:
    print(f'  {name:<23} {v:>9.4f} {t:>9.4f}')

# 2-feature non-triviality baseline
rf_simple = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_simple.fit(X_train[['recency','tx_count']], y_train)
simple_auc = roc_auc_score(y_test, rf_simple.predict_proba(X_test[['recency','tx_count']])[:,1])
print(f'\n2-feature baseline (recency+tx_count): {simple_auc:.4f}')
print(f'Full RF model:                          {rf_test_auc:.4f}')
print(f'Gain from feature engineering:          {rf_test_auc - simple_auc:+.4f}')


In [ ]:
# ROC curves for thesis figure
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, auc, color in [
    ('Random Forest',       rf_test_proba,  rf_test_auc,  '#2563EB'),
    ('XGBoost',             xgb_test_proba, xgb_test_auc, '#DC2626'),
    ('Logistic Regression', lr_test_proba,  lr_test_auc,  '#16A34A'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name}  (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves -- test set'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Save model and predictions
with open(DATA / 'best_model_rf.pkl', 'wb') as f:
    pickle.dump(rf, f)

X_train.to_csv(DATA / 'X_train.csv', index=False)
X_val.to_csv(DATA / 'X_val.csv', index=False)
X_test.to_csv(DATA / 'X_test.csv', index=False)
y_train.to_csv(DATA / 'y_train.csv', index=False)
y_val.to_csv(DATA / 'y_val.csv', index=False)
y_test.to_csv(DATA / 'y_test.csv', index=False)

pd.DataFrame({
    'client_id': df_test['client_id'].values,
    'churned':   y_test.values,
    'proba_rf':  rf_test_proba,
    'proba_xgb': xgb_test_proba,
    'proba_lr':  lr_test_proba,
}).to_csv(DATA / 'test_predictions.csv', index=False)

print('Saved: best_model_rf.pkl, X/y splits, test_predictions.csv')
